# bert-1
- run pipeline

In [ ]:
from nlp import ClimateBERTAnalyzer, analyze_reports

stats = analyze_reports('../data/reports/Baosteel')
# stats = analyze_reports("../data/reports")

# bert-2
- run vizualisations

In [ ]:
from nlp import ClimateBERTVisualizer, visualize_results

visualize_results("../cache", "../out")

✅ Loaded: 15 reports, 1 companies, 2013-2020

EXPORTING CSV FILES
   ✓ company_year.csv (8 rows)
   ✓ company_totals.csv (1 companies)
   ✓ yearly_industry.csv (8 years)
   ✓ funnel_company_year.csv (8 rows)

   📁 All CSVs saved to: ../out/

GENERATING PLOTS
   ✓ slide_main.png
   ✓ slide_sentiment_trend.png
   ✓ talk_score_trend.png
   ✓ funnel_trend.png
   ✓ talk_score_per_company.png
   ✓ per_company_components.png
   ✓ per_company_sentiment.png
   ✓ sentiment_all_companies.png
   ✓ n0_funnel.png
   ✓ n0_quality_comparison.png
   ✓ n0_per_company.png
   ✓ n0_gap_analysis.png

   📁 All plots saved to: ../out/

   Generating word frequency plots...

   📊 ALL CHUNKS (top 30):
   environment(1232), iron(766), management(736), development(720), energy(628), technology(609), products(607), protection(559), production(525), industry(522), green(473), emission(453), system(428), reduction(421), base(410)

   🌱 OPPORTUNITY chunks (top 15):
   environment(809), development(574), technology(52

# rag 1

## Reload modules

run when changing code in .py file

In [2]:
%load_ext autoreload
%autoreload 2
# %aimport nlp.rag_1

# import os
# print(os.getcwd())

## Model Testing

Test which models work for extraction (format compliance + quality).

In [1]:
from nlp.rag_test import save_test_results
from nlp.rag_test import test_models, compare_extractions
from nlp import RAGConfig

# Available models (for reference):
# Groq:   mixtral-8x7b-32768, gemma2-9b-it
# Ollama: gemma3:4b

MODELS = [
    # Groq (free tier: 6k TPM - use small batch_size)
    RAGConfig(llm_provider="groq", model="llama-3.1-8b-instant", ctx=128000, batch_size=3),
    # RAGConfig(llm_provider="groq", model="llama-3.3-70b-versatile", ctx=128000, batch_size=5),

    # Ollama (local - no rate limits)
    # Tested: gemma3:4b ctx=4096 works but slow (108s), qwen2.5:3b fails format
    # RAGConfig(llm_provider="ollama", model="qwen2.5:3b", ctx=4096),  # 1.9GB - FAIL format
    # RAGConfig(llm_provider="ollama", model="gemma3:4b", ctx=4096),   # 3.3GB - PASS but slow

    # Testing: smaller ctx to reduce VRAM pressure on 4GB GPU
    # RAGConfig(llm_provider="ollama", model="llama3.2:3b", ctx=4096),  # 2.0GB
    # RAGConfig(llm_provider="ollama", model="llama3.2:3b", ctx=4096),  # 2.0GB
    # RAGConfig(llm_provider="ollama", model="llama3.1:8b ", ctx=4096),  # 2.0GB

    # RAGConfig(llm_provider="ollama", model="qwen3:4b", ctx=2048),     # 2.5GB
]

results = test_models(MODELS, skip_extraction=False)
compare_extractions(results)
save_test_results(results) # Save test results (same format as full pipeline: CSV + stats.json with prompts)

RAG MODEL TESTING (1 models) - 006/2024

--- groq/llama-3.1-8b-instant (ctx=128,000 → 3 chunks) ---
    Format: PASS (0.3s)
    Output: - High cost of green hydrogen
- Infrastructure limitations
- Regulatory uncertainty...
✓ RAG Pipeline initialized (GPU: NVIDIA T1200 Laptop GPU (3.9GB))
✓ Loaded 15593 chunks from 15 companies (../cache)
Loading Groq: llama-3.1-8b-instant
    B done: 2230 found in 44 batches (653.1s)          
    M done: 1123 found in 44 batches (675.4s)          
    Extraction: 2230B, 1123M (1328.5s)

Model                                    Fmt    Time    B     M    
----------------------------------------------------------------------
groq/llama-3.1-8b-instant                PASS   0.3s    2230  1123 
Need 2+ models with extraction results to compare
✓ Test run saved to ../out/0211_groq_llama-3.1-8b-instant/


## Run Extraction

Configure and run the full extraction pipeline.

In [ ]:
from nlp import load_pipeline, RAGConfig

config = RAGConfig(llm_provider="ollama", model="llama3.2:3b", ctx=4096)

pipeline = load_pipeline(config)
# pipeline.print_chunk_overview()

In [3]:
# Extract all companies (saves to out/)
results = pipeline.extract_all_companies()



EXTRACTION RUN
  LLM: ollama/llama3.2:3b
  Context: 4,096 tokens → 8 chunks/batch
  Chunks: 15593 (1750 avg chars, 38-19629 range)
  Groups: 132 | Est. LLM calls: 4016


Extracting: ArcelorMittal (001)
Years: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Chunks: 4344 | Batches: 549 × 2 (barriers+motivators) = 1098 LLM calls


  001:   0%|          | 0/13 [00:00<?, ?it/s]

Loading Ollama: llama3.2:3b


  001:   8%|▊         | 1/13 [20:21<4:04:22, 1221.86s/it]

    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_007' not found in lookup
    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_522' not found in lookup
    ⚠️ chunk_id '012_203' not found in lookup


...

  016:  83%|████████▎ | 10/12 [1:11:22<18:27, 553.80s/it]

    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_007' not found in lookup
    ⚠️ chunk_id '012_043' not found in lookup
    ⚠️ chunk_id '012_005' not found in lookup
    ⚠️ chunk_id '012_074' not found in lookup
    ⚠️ chunk_id '012_004' not found in lookup
    ⚠️ chunk_id '012_096' not found in lookup
    ⚠️ chunk_id '012_038' not found in lookup
    ⚠️ chunk_id '012_007' not found in lookup
    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_007' not found in lookup
    ⚠️ chunk_id '012_065' not found in lookup


  016:  92%|█████████▏| 11/12 [1:21:31<09:30, 570.72s/it]

    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_007' not found in lookup
    ⚠️ chunk_id '012_021' not found in lookup
    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_006' not found in lookup
    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_007' not found in lookup
    ⚠️ chunk_id '012_018' not found in lookup
    ⚠️ chunk_id '012_079' not found in lookup
    ⚠️ chunk_id '012_140' not found in lookup
    ⚠️ chunk_id '012_103' not found in lookup
    ⚠️ chunk_id '012_088' not found in lookup
    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_007' not found in lookup
    ⚠️ chunk_id '012_059' not found in lookup
    ⚠️ chunk_id '012_062' not found in lookup
    ⚠️ chunk_id '012_083' not found in lookup
    ⚠️ chunk_id '012_006' not found in lookup


    ⚠️ chunk_id '012_074' not found in lookup
    ⚠️ chunk_id '012_078' not found in lookup
    ⚠️ chunk_id '012_042' not found in lookup
    ⚠️ chunk_id '012_013' not found in lookup
    ⚠️ chunk_id '012_139' not found in lookup
    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_007' not found in lookup
    ⚠️ chunk_id '012_066' not found in lookup
    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_007' not found in lookup
    ⚠️ chunk_id '012_114' not found in lookup
    ⚠️ chunk_id '012_140' not found in lookup
    ⚠️ chunk_id '012_062' not found in lookup
    ⚠️ chunk_id '012_003' not found in lookup
    ⚠️ chunk_id '012_007' not found in lookup
    ⚠️ chunk_id '012_006' not found in lookup
    ⚠️ chunk_id '012_030' not found in lookup
  ✓ Extracted 103 barriers (53% ID-matched), 199 motivators (60% ID-matched)

✓ EXTRACTION COMPLETE
  Time: 68543.4s (1142.4min)
  Results: 1315 barriers, 1344 motivators
 

In [4]:
# Display results for a specific company
company_id = list(results.keys())[0]  # First company
df_barriers, df_motivators = results[company_id]
pipeline.display_results(company_id, df_barriers, df_motivators)


BARRIERS - 001 (392 items)
+-----+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# rag 2

In [ ]:
# from bertopic.representation import OpenAI,LlamaCPP
from nlp import TopicModelConfig, run_topic_modeling_pipeline

# Set True to ignore cached model/embeddings and retrain from scratch
FORCE_RETRAIN = True

config = TopicModelConfig(
    # Embedding model
    # embedding_model="sentence-transformers/all-mpnet-base-v2",
    embedding_model="BAAI/bge-small-en-v1.5",

    batch_size=64,

    # UMAP (dimensionality reduction)
    umap_n_neighbors=30,
    umap_n_components=15,
    umap_min_dist=0.05,
    umap_metric='cosine',
    umap_random_state=42,

    # HDBSCAN (clustering)
    hdbscan_min_cluster_size=10,  # Higher = fewer topics
    hdbscan_min_samples=2,        # Lower = less outliers
    hdbscan_metric='euclidean',
    hdbscan_cluster_selection_method='leaf',  # 'eom' (inclusive) or 'leaf' (tight, more cleanup required)

    # Vectorizer (c-TF-IDF)
    vectorizer_ngram_range=(1, 2),
    vectorizer_min_df=1,
    vectorizer_max_df=0.95,

    # Topic representation
    mmr_diversity=0.2, # 0 - pure relevance, redundant/simi word ... 1 - pure diverse. max diff word
    top_n_words=10,
    nr_topics=10,  # Set None for auto, or int to reduce post-hoc
    calculate_probabilities=True,

    # Outlier reduction (post-hoc)
    reduce_outliers=False,  # Reassign outliers to nearest topic
    reduce_outliers_strategy='embeddings',  # 'embeddings', 'c-tf-idf', or 'distributions'

    # Visualization UMAP (separate 2D projection)
    viz_umap_n_neighbors=10,
    viz_umap_n_components=2,
    viz_umap_min_dist=0.0,

    # LLM for topic labeling
    ollama_model="gemma3:4b",  # Fast + follows format. Avoid qwen3 (hidden thinking = slow)
    ollama_base_url="http://localhost:11434",
    llm_temperature=0.0,

    # Misc
    verbose=True,
    embeddings_cache_path=None,
)

results = run_topic_modeling_pipeline(
    data_folder="../out",
    output_folder="../out/topics5",
    config=config,
    force_retrain=FORCE_RETRAIN
)
# Access results
barriers_df = results['barriers']['df']
motivators_df = results['motivators']['df']

In [ ]:
motivators_df